In [12]:
import duckdb
import pandas as pd

# Connect to existing database
con = duckdb.connect(r"C:\Users\shaog\OneDrive\Documents\DSE3101\mind-the-data-gap\data.duckdb")

In [13]:
# Check tables
print(con.execute("SHOW TABLES").fetchdf())

          name
0  abt_pt_jrny
1  abt_pt_ride
2     bus_stop
3      pt_jrny
4      pt_ride
5   vls_marker


## Cleaning pt_ride
1. Drop incomplete data
2. Pick a representative weekday (13/2/2025)
3. Drop irrelevant columns
4. Select relevant concession types

In [14]:
# Check columns of pt_ride first
print(con.execute("DESCRIBE pt_ride").fetchdf())

           column_name column_type null   key default extra
0               BIZ_DT        DATE  YES  None    None  None
1          BUS_SVC_NUM      BIGINT  YES  None    None  None
2              CRD_NUM     VARCHAR  YES  None    None  None
3      DEST_LOC_ID_NUM      BIGINT  YES  None    None  None
4             ENTRY_DT        DATE  YES  None    None  None
5             ENTRY_TM        TIME  YES  None    None  None
6              EXIT_DT        DATE  YES  None    None  None
7              EXIT_TM        TIME  YES  None    None  None
8          JRNY_ID_NUM      BIGINT  YES  None    None  None
9      ORIG_LOC_ID_NUM      BIGINT  YES  None    None  None
10  PATRON_CATG_ID_NUM      BIGINT  YES  None    None  None
11              PAY_CD      BIGINT  YES  None    None  None
12       RIDE_DISC_AMT      DOUBLE  YES  None    None  None
13    RIDE_DIST_KM_CNT      DOUBLE  YES  None    None  None
14       RIDE_FARE_AMT      DOUBLE  YES  None    None  None
15         RIDE_ID_NUM      BIGINT  YES 

In [ ]:
# Select representative weekday and drop incomplete rows
cleaned_pt_ride = con.execute("""
    SELECT 
        CRD_NUM,
        JRNY_ID_NUM,
        BUS_SVC_NUM,
        ENTRY_DT,
        ENTRY_TM,
        EXIT_DT,
        EXIT_TM,
        ORIG_LOC_ID_NUM,
        DEST_LOC_ID_NUM,
        TRNSPT_MODE_CD,
        PATRON_CATG_ID_NUM,
        RIDE_FARE_AMT,
        RIDE_DIST_KM_CNT,
        RIDE_MIN_CNT,
        RIDE_ST_CD
    FROM pt_ride
    WHERE BIZ_DT = '2025-02-13'
      AND CRD_NUM IS NOT NULL
      AND ORIG_LOC_ID_NUM IS NOT NULL
      AND DEST_LOC_ID_NUM IS NOT NULL
      AND ENTRY_TM IS NOT NULL
      AND EXIT_TM IS NOT NULL
      AND RIDE_FARE_AMT IS NOT NULL
      AND RIDE_MIN_CNT IS NOT NULL
      AND RIDE_ST_CD = 1
""").fetchdf()

print(cleaned_pt_ride.shape) # (3727069, 15)
# print(cleaned_pt_ride.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3727069, 15)


In [20]:
print(cleaned_pt_ride['PATRON_CATG_ID_NUM'].value_counts().sort_index())

PATRON_CATG_ID_NUM
0        837
1    1665312
2      12397
3     728162
4    1161122
5      15999
6     143240
Name: count, dtype: int64


In [21]:
con.close()